<div style="border-style:groove;border-width:thin;padding:10px">
    <h3>EJERCICIO 1:</h3>
</div>

In [2]:
# Datos y preprocesamiento
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Modelos de Clasificación
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Modelos de Regresión
from sklearn.svm import SVR, LinearSVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Métricas de Clasificación
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Métricas de Regresión
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Todas las librerías importadas correctamente")

✅ Todas las librerías importadas correctamente


In [3]:
import pandas as pd
peliculas = pd.read_csv('peliculas.csv')
peliculas.head(2)

,budget,genres,homepage,id,original_language,original_title,overview,popularity,production_companies,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"['Action', 'Adventure', 'Fantasy', 'Science Fi...",http://www.avatarmovie.com/,19995,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"['Ingenious Film Partners', 'Twentieth Century...",2009-12-10,2787965087,162.0,"['English', 'Español']",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"['Adventure', 'Fantasy', 'Action']",http://disney.go.com/disneypictures/pirates/,285,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"['Walt Disney Pictures', 'Jerry Bruckheimer Fi...",2007-05-19,961000000,169.0,['English'],Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [4]:
import ast
peliculas['genres'] = peliculas['genres'].apply(ast.literal_eval)
peliculas['production_companies'] = peliculas['production_companies'].apply(ast.literal_eval)
peliculas['spoken_languages'] = peliculas['spoken_languages'].apply(ast.literal_eval)

    

In [29]:
peliculas.columns

Index(['budget', 'popularity', 'revenue', 'runtime', 'vote_average',
       'vote_count', 'status_Post Production', 'status_Released',
       'status_Rumored'],
      dtype='object')

In [6]:
peliculas['status'].unique()

array(['Released', 'Post Production', 'Rumored'], dtype=object)

In [7]:
#Borrado de columnas no utiles, datos que no son unicos o no aportan una información importante
""" 
columnas borradas
Genres: el genero no aporta información importante y es dificil de tratar, e incluso podría provocar algun sesgo en base al publico que vote
production_companies: La productoras no creo que influyan  tanto, pueden influir en la calidad final de la pelicula, pero como tampoco hay garantia de que hagan todo bien o todo mal, me la cargo
spoken_languages y original_language: Las lenguas habladas en la pelicula no lo veo dato relevante debido a la existencia del doblaje
release_date: La fecha de salida no la veo importante porque no nos deja nada en claro, no creo que tenga releveancia que despendiendo del año mejore o empeore la popularidad
El resto como title o homepage siento que son datos que no se repiten por lo que no nos van a dar datos importantes

En caso de que las columnas de listas fuesen importante, trataria de coger el dato principal del genero o de la productora, en el caso de las lenguas, me las cargaria todas
"""
peliculas.drop(columns=['id','homepage','overview','tagline','genres','production_companies','spoken_languages','original_language','title','release_date','original_title'], axis=1, inplace=True)

In [8]:
peliculas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   budget        4803 non-null   int64  
 1   popularity    4803 non-null   float64
 2   revenue       4803 non-null   int64  
 3   runtime       4801 non-null   float64
 4   status        4803 non-null   object 
 5   vote_average  4803 non-null   float64
 6   vote_count    4803 non-null   int64  
dtypes: float64(3), int64(3), object(1)
memory usage: 262.8+ KB


In [9]:
#hay pocos nulos, asi que hago un dropna directo
peliculas = peliculas.dropna()

In [10]:
#Muestra de que los datos no se han visto muy afectados
peliculas.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4801 entries, 0 to 4802
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   budget        4801 non-null   int64  
 1   popularity    4801 non-null   float64
 2   revenue       4801 non-null   int64  
 3   runtime       4801 non-null   float64
 4   status        4801 non-null   object 
 5   vote_average  4801 non-null   float64
 6   vote_count    4801 non-null   int64  
dtypes: float64(3), int64(3), object(1)
memory usage: 300.1+ KB


In [11]:
peliculas.corr(numeric_only=True)['vote_average'].abs().sort_values(ascending=False)[1:]

runtime       0.375046
vote_count    0.313423
popularity    0.274171
revenue       0.197286
budget        0.092728
Name: vote_average, dtype: float64

In [12]:
#Adaptacion de columna status
peliculas = pd.get_dummies(peliculas,dtype=int)

In [13]:
#Preparacion de X e Y
X = peliculas.drop(columns=['vote_average'])
y = peliculas['vote_average']
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

In [15]:
#Entrenamiento con svm lineal
svr_lnr = SVR(kernel="linear", C=11)

svr_lnr.fit(X_train,y_train)
y_pred_lnr = svr_lnr.predict(X_test)

print(r2_score(y_test, y_pred_lnr))


0.19968020817900645


In [16]:
#Entrenamiento con rbf
svr_rbf = SVR(kernel="rbf", C=1000)

svr_rbf.fit(X_train,y_train)
y_pred_rbf = svr_rbf.predict(X_test)

print(r2_score(y_test, y_pred_rbf))

"""
Con un C muy grande el R2 se reduce mucho, pero, no lo ajustaria tanto ya que puede provocar un overfitting, con C 5000 logré un 0,04, 
pero prefiero usar un C como 1000 que da buen resultado y no tenderá a usar un sobreajuste
"""


0.19445026616588812


'\nCon un C muy grande el R2 se reduce mucho, pero, no lo ajustaria tanto ya que puede provocar un overfitting, con C 5000 logré un 0,04, \npero prefiero usar un C como 1000 que da buen resultado y no tenderá a usar un sobreajuste\n'

In [17]:
#Entrenamiento con poly

svr_poly = SVR(kernel="poly", degree=1, C=11, coef0=0.1)

svr_poly.fit(X_train,y_train)
y_pred_poly = svr_poly.predict(X_test)

print(r2_score(y_test, y_pred_poly))

0.1996691608395963


In [18]:
#Entrenamiento con decision trees

tree_reg = DecisionTreeRegressor(random_state=0)

tree_reg.fit(X_train, y_train)
y_pred_tree = tree_reg.predict(X_test)

print(r2_score(y_test, y_pred_tree))

from sklearn.tree import export_graphviz
# For a regressor there is no `classes_` attribute, so omit `class_names`.
export_graphviz(
    tree_reg,
    out_file="./peliculas.dot",
    feature_names=list(peliculas.drop(['vote_average'], axis=1).columns),
    rounded=True,
    filled=True
)

# If "dot: command not found" occurs, install graphviz in the system:
# sudo apt install graphviz
!dot -Tpng peliculas.dot -o peliculas.png
"""
El R2 mas pequeño es sin parametros, pero claro, probablemente tienda mucho al overfitting, el max depth anda entre 15 y 20,
por lo que el dato mas importante para controlar esto es con min_samples_leaf
"""



0.14737223685323886
dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.122275 to fit


'\nEl R2 mas pequeño es sin parametros, pero claro, probablemente tienda mucho al overfitting, el max depth anda entre 15 y 20,\npor lo que el dato mas importante para controlar esto es con min_samples_leaf\n'

In [19]:
#Entrenamiento con ramdon forest
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

print("R^2 Score:", rf.score(X_test, y_test))

R^2 Score: 0.5319737192072935


In [20]:
"""
Cuestion teorica: ¿Como hacer una pelicula que guste a la gente?
En base a los datos utilizados para entrenar al modelo, trataria de invertir un gran presupuesto para realizar la pelicula, 
y mirar el tiempo de pelicula mas ajustado a lo que la gente suele ver, por ejemplo, 90 minutos, mirando los datos de las peliculas con
mayor popularidad también, para conseguir una alta

En base a otros datos que no he usado para entrenar, trataría de contratar a la productora con mejores resultados de popularidad, 
y mirar también los generos mas populares de peliculas, por lo que esto seguiría relacionado a la popularidad, 
ya otras cuestiones como casting, se van de este ejercicio
"""

'\nCuestion teorica: ¿Como hacer una pelicula que guste a la gente?\nEn base a los datos utilizados para entrenar al modelo, trataria de invertir un gran presupuesto para realizar la pelicula, \ny mirar el tiempo de pelicula mas ajustado a lo que la gente suele ver, por ejemplo, 90 minutos, mirando los datos de las peliculas con\nmayor popularidad también, para conseguir una alta\n\nEn base a otros datos que no he usado para entrenar, trataría de contratar a la productora con mejores resultados de popularidad, \ny mirar también los generos mas populares de peliculas, por lo que esto seguiría relacionado a la popularidad, \nya otras cuestiones como casting, se van de este ejercicio\n'

<div style="border-style:groove;border-width:thin;padding:10px">
    <h3>EJERCICIO 2:</h3>
</div>

In [119]:
df_students = pd.read_csv('estudiantes.csv')

df_students.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8%,1.4%,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9%,-0.3%,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8%,1.4%,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4%,-0.8%,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9%,-0.3%,0.79,Graduate


In [120]:
df_students.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance	                     4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualification                   

In [121]:
df_students['Age at enrollment'].unique()

array(['twenty', 'nineteen', 'forty-five', 'fifty', 'eighteen',
       'twenty-two', 'twenty-one', 'thirty-four', 'thirty-seven',
       'forty-three', 'fifty-five', 'thirty-nine', 'twenty-nine',
       'twenty-four', 'twenty-seven', 'twenty-three', 'twenty-six',
       'thirty-three', 'thirty-five', 'twenty-five', 'forty-four',
       'thirty-six', 'forty-seven', 'twenty-eight', 'thirty-eight',
       'thirty', 'thirty-one', 'thirty-two', 'forty', 'forty-two',
       'forty-eight', 'forty-nine', 'forty-six', 'forty-one', 'seventy',
       'sixty', 'fifty-three', 'fifty-one', 'fifty-two', 'fifty-four',
       'sixty-one', 'fifty-eight', 'fifty-nine', '17', 'fifty-seven',
       'sixty-two'], dtype=object)

In [122]:
prueba = "5.2%"
print(prueba.split(sep="%"))

['5.2', '']


In [123]:
df_students.replace({"Target": {"Dropout": 0, "Graduate": 1, "Enrolled": 2}}, inplace=True)

/tmp/ipykernel_42216/205857960.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_students.replace({"Target": {"Dropout": 0, "Graduate": 1, "Enrolled": 2}}, inplace=True)


In [124]:
#Voy a poner la columna Price con valores float en vez de string
inflation_rate = []
unemployement_rate = []
for index,row in df_students.iterrows():
    inflation_rate.append(float(row['Inflation rate'].split("%")[0]))
    unemployement_rate.append(float(row['Unemployment rate'].split("%")[0]))

df_students['Inflation rate'] = inflation_rate
df_students['Unemployment rate'] = unemployement_rate    
df_students.info()
df_students.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance	                     4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualification                   

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,0
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,1
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,0
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,1
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,1


In [ ]:
#Hay 2 columnas con muchos nulos que son numeros que no nos dan información interesante asi que me las cargo
df_students.drop(columns=["Father's occupation", "Mother's occupation"], inplace=True)

#Para la pregunta 6
#df_students.drop(columns=["Nacionality", "Gender"], inplace=True)
#df_students.drop(columns=["Nacionality"], inplace=True)

In [126]:
df_students.corr(numeric_only=True)['Target'].abs().sort_values(ascending=False)[1:]

Curricular units 2nd sem (grade)                  0.429214
Curricular units 2nd sem (approved)               0.351135
Curricular units 1st sem (grade)                  0.349652
Tuition fees up to date                           0.342121
Curricular units 1st sem (approved)               0.290243
Curricular units 2nd sem (evaluations)            0.194412
Debtor                                            0.154802
Curricular units 1st sem (evaluations)            0.125278
Gender                                            0.118454
Application mode                                  0.116928
Scholarship holder                                0.114517
Mother's qualification                            0.075941
Marital status                                    0.074310
Displaced                                         0.070649
Daytime/evening attendance\t                      0.066439
Curricular units 2nd sem (enrolled)               0.060670
Curricular units 1st sem (enrolled)               0.0520

In [127]:
df_students = pd.get_dummies(df_students, dtype=int)

In [128]:
X = df_students.drop(['Target'],axis=1)
y = df_students['Target'].to_frame()
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

In [129]:
svc_rbf = SVC(kernel="rbf", C=1)
svc_rbf.fit(X_train,y_train)
y_pred_rbf = svc_rbf.predict(X_test)
print(accuracy_score(y_test, y_pred_rbf))

/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


0.7613019891500904


In [130]:
svc_linear = SVC(kernel="linear", C=1)
svc_linear.fit(X_train,y_train)
y_pred_linear = svc_linear.predict(X_test)
print(accuracy_score(y_test, y_pred_linear))

/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


0.7585895117540687


In [131]:
svc_poly = SVC(kernel="poly", C=11, degree=1, coef0=1)
svc_poly.fit(X_train,y_train)
y_pred_poly = svc_poly.predict(X_test)
print(accuracy_score(y_test, y_pred_poly))

/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


0.7622061482820977


In [132]:

tree_cla = DecisionTreeClassifier(random_state=0,max_depth=9,min_samples_split=40)
tree_cla.fit(X_train, y_train)
y_pred_tree = tree_cla.predict(X_test)

print(accuracy_score(y_test, y_pred_tree))

0.7296564195298373


In [133]:
rf = RandomForestClassifier(n_estimators=700, random_state=42, n_jobs=-1,max_depth=25)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Accuracy: 0.8020


In [ ]:
"""
Columnas que podria borrar: Quitaria la nacionalidad y el genero, para evitar crear sesgos principalemente, hay algunas columnas que tengo en duda pero no se lo que significan
por lo que prefiero no borrarlas.

Hecha la prueba, puedo ver que los resultados bajan, por lo que creo que el genero aqui si influye, además es de la que mas correlación negativa tiene, voy a probar solo nacionalidad

También otra opción es quitar aquellas con menor correlación que no nos aportan tanta info

Quitando la nacionalidad los resultados mejoran un minimo si no recuerdo mal
"""

'\nColumnas que podria borrar: Quitaria la nacionalidad y el genero, para evitar crear sesgos principalemente, hay algunas columnas que tengo en duda pero no se lo que significan\npor lo que prefiero no borrarlas.\n\nHecha la prueba, puedo ver que los resultados bajan, por lo que creo que el genero aqui si influye, además es de la que mas correlación negativa tiene, voy a probar solo nacionalidad\n\nTambién otra opción es quitar aquellas con menor correlación que no nos aportan tanta info\n'